# 02 — Analyse exploratoire (EDA)
## Coût de la vie et inflation au Sénégal (2018–2026)

Toutes les figures sont sauvegardées dans `reports/figures/` pour le rapport et
le README. On répond aux questions clés :
1. Comment évolue l'inflation au Sénégal (2018–2026) ?
2. Quels produits sont les plus inflationnistes ?
3. Alimentaire vs énergie : qui tire les prix ?
4. Quelles régions sont les plus touchées ?
5. Comment évolue le coût du panier et le pouvoir d'achat ?


In [ ]:

import os, warnings, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["figure.dpi"] = 110

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)
print("Racine projet :", PROJ)


In [ ]:

ihpc_nat = pd.read_csv(os.path.join(PROC, "ihpc_national.csv"), parse_dates=["date"])
ihpc_reg = pd.read_csv(os.path.join(PROC, "ihpc_regional.csv"), parse_dates=["date"])
fact_ihpc = pd.read_csv(os.path.join(PROC, "fact_ihpc.csv"), parse_dates=["date"])
fact_prix = pd.read_csv(os.path.join(PROC, "fact_prix.csv"), parse_dates=["date"])
panier_nat = pd.read_csv(os.path.join(PROC, "panier_national.csv"), parse_dates=["date"])
panier_reg = pd.read_csv(os.path.join(PROC, "panier_regional.csv"), parse_dates=["date"])
dim_div = pd.read_csv(os.path.join(PROC, "dim_division.csv"))
print("Données chargées.")


### 1. Évolution de l'IHPC national et de l'inflation (glissement annuel)

In [ ]:

fig, ax1 = plt.subplots()
ax1.plot(ihpc_nat["date"], ihpc_nat["indice_global"], color="#1f4e79", lw=2, label="IHPC global (base 100=2023)")
ax1.set_ylabel("IHPC (base 100 = 2023)", color="#1f4e79")
ax2 = ax1.twinx()
ax2.plot(ihpc_nat["date"], ihpc_nat["var_annuelle_pct"], color="#c0392b", lw=1.6, label="Inflation (glissement annuel %)")
ax2.axhline(0, color="grey", lw=.6)
ax2.set_ylabel("Inflation glissement annuel (%)", color="#c0392b")
peak = ihpc_nat.loc[ihpc_nat["var_annuelle_pct"].idxmax()]
ax2.annotate(f"Pic {peak['var_annuelle_pct']:.1f}%\n{peak['date']:%b %Y}",
             xy=(peak["date"], peak["var_annuelle_pct"]),
             xytext=(peak["date"], peak["var_annuelle_pct"]+2),
             arrowprops=dict(arrowstyle="->", color="#c0392b"), color="#c0392b", fontsize=9)
plt.title("Sénégal — IHPC et inflation en glissement annuel (2018–2026)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "01_inflation_nationale.png"), bbox_inches="tight")
plt.close(fig); print("→ 01_inflation_nationale.png")


### 2. Inflation par division — heatmap (année × division)

In [ ]:

fi = fact_ihpc.copy()
fi["annee"] = fi["date"].dt.year
# inflation moyenne annuelle par division (national : moyenne des régions)
piv = (fi.groupby(["division","annee"])["var_annuelle_pct"].mean()
         .reset_index().pivot(index="division", columns="annee", values="var_annuelle_pct"))
piv = piv.reindex(dim_div.set_index("division").index)  # ordre COICOP
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(piv, annot=True, fmt=".1f", cmap="RdYlGn_r", center=0,
            cbar_kws={"label":"Inflation annuelle moyenne (%)"}, ax=ax)
ax.set_title("Inflation par division de consommation et par année")
ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "02_heatmap_divisions.png"), bbox_inches="tight")
plt.close(fig); print("→ 02_heatmap_divisions.png")


### 3. Produits les plus inflationnistes (hausse cumulée 2018 → 2026)

In [ ]:

fp = fact_prix.copy()
nat_prod = fp.groupby(["produit","date"])["prix_moyen"].mean().reset_index()
deb = nat_prod.sort_values("date").groupby("produit").first()["prix_moyen"]
fin = nat_prod.sort_values("date").groupby("produit").last()["prix_moyen"]
hausse = ((fin/deb - 1)*100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette("flare", len(hausse))
ax.barh(hausse.index[::-1], hausse.values[::-1], color=colors)
for i, v in enumerate(hausse.values[::-1]):
    ax.text(v+0.5, i, f"{v:.0f}%", va="center", fontsize=8)
ax.set_title("Hausse cumulée des prix par produit (2018 → 2026)")
ax.set_xlabel("Variation cumulée (%)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "03_top_produits.png"), bbox_inches="tight")
plt.close(fig); print("→ 03_top_produits.png")
hausse.round(1).to_frame("hausse_cumulee_%")


### 4. Alimentaire vs énergie vs global

In [ ]:

fig, ax = plt.subplots()
ax.plot(ihpc_nat["date"], ihpc_nat["var_annuelle_pct"], label="Global", color="#1f4e79", lw=2)
ax.plot(ihpc_nat["date"], ihpc_nat["var_alim_annuelle_pct"], label="Alimentaire", color="#27ae60", lw=1.6)
ax.plot(ihpc_nat["date"], ihpc_nat["var_energie_annuelle_pct"], label="Énergie", color="#e67e22", lw=1.6)
ax.axhline(0, color="grey", lw=.6); ax.legend()
ax.set_title("Inflation en glissement annuel : global vs alimentaire vs énergie")
ax.set_ylabel("%")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "04_alim_vs_energie.png"), bbox_inches="tight")
plt.close(fig); print("→ 04_alim_vs_energie.png")


### 5. Comparaison régionale de l'inflation

In [ ]:

fig, ax = plt.subplots()
for reg, g in ihpc_reg.groupby("region"):
    ax.plot(g["date"], g["var_annuelle_pct"], label=reg, lw=1.3)
ax.axhline(0, color="grey", lw=.6); ax.legend(ncol=3, fontsize=8)
ax.set_title("Inflation en glissement annuel par région")
ax.set_ylabel("%")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "05_inflation_regions.png"), bbox_inches="tight")
plt.close(fig)

# Heatmap région × année
ihpc_reg["annee"] = ihpc_reg["date"].dt.year
pr = ihpc_reg.groupby(["region","annee"])["var_annuelle_pct"].mean().reset_index().pivot(
    index="region", columns="annee", values="var_annuelle_pct")
fig, ax = plt.subplots(figsize=(11,4))
sns.heatmap(pr, annot=True, fmt=".1f", cmap="RdYlGn_r", center=0, ax=ax,
            cbar_kws={"label":"Inflation annuelle (%)"})
ax.set_title("Inflation annuelle moyenne par région"); ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "06_heatmap_regions.png"), bbox_inches="tight")
plt.close(fig); print("→ 05_inflation_regions.png, 06_heatmap_regions.png")


### 6. Distribution des prix par produit (boxplots)

In [ ]:

top_alim = ["Riz brisé ordinaire","Huile végétale raffinée","Sucre cristallisé",
            "Oignon local","Pomme de terre","Poisson frais (sardinelle)"]
sub = fact_prix[fact_prix["produit"].isin(top_alim)]
fig, ax = plt.subplots(figsize=(11,5))
sns.boxplot(data=sub, x="produit", y="prix_moyen", ax=ax, palette="Set2")
ax.set_title("Distribution des prix mensuels (produits alimentaires clés)")
ax.set_xlabel(""); ax.set_ylabel("Prix (FCFA)")
plt.xticks(rotation=20, ha="right")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "07_boxplot_prix.png"), bbox_inches="tight")
plt.close(fig); print("→ 07_boxplot_prix.png")


### 7. Décomposition de la tendance (STL) de l'IHPC national

In [ ]:

from statsmodels.tsa.seasonal import STL
s = ihpc_nat.set_index("date")["indice_global"].asfreq("MS")
res = STL(s, period=12, robust=True).fit()
fig = res.plot(); fig.set_size_inches(11, 8)
fig.suptitle("Décomposition STL de l'IHPC national", y=1.01)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "08_decomposition_stl.png"), bbox_inches="tight")
plt.close(fig); print("→ 08_decomposition_stl.png")


### 8. Coût du panier de base & pouvoir d'achat

In [ ]:

fig, ax1 = plt.subplots()
ax1.plot(panier_nat["date"], panier_nat["cout_panier"]/1000, color="#8e44ad", lw=2, label="Coût nominal")
ax1.plot(panier_nat["date"], panier_nat["cout_panier_reel"]/1000, color="#16a085", lw=1.6, ls="--", label="Coût réel (FCFA 2023)")
ax1.set_ylabel("Coût du panier (milliers FCFA / mois)"); ax1.legend(loc="upper left")
ax1.set_title("Coût du panier de base — nominal vs réel")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "09_cout_panier.png"), bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots()
ax.plot(ihpc_nat["date"], ihpc_nat["pouvoir_achat_index"], color="#c0392b", lw=2)
ax.axhline(100, color="grey", lw=.6, ls="--")
ax.fill_between(ihpc_nat["date"], ihpc_nat["pouvoir_achat_index"], 100,
                where=ihpc_nat["pouvoir_achat_index"]<100, color="#c0392b", alpha=.15)
ax.set_title("Indice de pouvoir d'achat (base 100 = 2023)")
ax.set_ylabel("Indice")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "10_pouvoir_achat.png"), bbox_inches="tight")
plt.close(fig)
perte = (1 - ihpc_nat["pouvoir_achat_index"].iloc[-1]/100)*100
print(f"→ 09_cout_panier.png, 10_pouvoir_achat.png")
print(f"Érosion du pouvoir d'achat depuis 2023 : {perte:.1f}%")


### 9. Contribution des divisions à l'inflation (dernière année)

In [ ]:

last = ihpc_nat["date"].max()
y1 = last - pd.DateOffset(years=1)
fi = fact_ihpc.copy()
nat = fi.groupby(["date","division","division_code"])["indice"].mean().reset_index()
piv = nat.pivot_table(index="date", columns=["division_code","division"], values="indice")
contrib = []
poids = dim_div.set_index("division_code")["poids_frac"]
for (code, lib) in piv.columns:
    try:
        i_last = piv[(code,lib)].asof(last); i_prev = piv[(code,lib)].asof(y1)
        var = (i_last/i_prev - 1)
        contrib.append((lib, var*poids[code]*100))
    except Exception:
        pass
cdf = pd.DataFrame(contrib, columns=["division","contribution_pts"]).sort_values("contribution_pts")
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(cdf["division"], cdf["contribution_pts"],
        color=["#c0392b" if v>0 else "#2980b9" for v in cdf["contribution_pts"]])
ax.set_title(f"Contribution des divisions à l'inflation ({y1:%b %Y} → {last:%b %Y})")
ax.set_xlabel("Contribution (points de %)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "11_contribution_inflation.png"), bbox_inches="tight")
plt.close(fig); print("→ 11_contribution_inflation.png")
cdf.round(2)


### 10. Carte choroplèthe du Sénégal
On projette les indicateurs des **6 zones de collecte** sur les **14 régions
administratives** (chaque région est rattachée à sa zone IHPC) à partir d'un
GeoJSON officiel (geoBoundaries ADM1).

In [ ]:

import json
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Polygon as MplPoly

geo_path = os.path.join(PROJ, "data", "geo", "senegal_regions.geojson")
geo = json.load(open(geo_path, encoding="utf-8"))

# Rattachement des 14 régions administratives aux 6 zones de collecte de l'IHPC
ZONE_MAP = {
    "Dakar":"Dakar", "Thies":"Thiès", "Diourbel":"Diourbel",
    "Louga":"Saint-Louis", "Saint Louis":"Saint-Louis", "Matam":"Saint-Louis",
    "Fatick":"Kaolack", "Kaolack":"Kaolack", "Kaffrine":"Kaolack",
    "Kolda":"Kolda", "Sedhiou":"Kolda", "Ziguinchor":"Kolda",
    "Tambacounda":"Kolda", "Kedougou":"Kolda",
}

def rings_of(geom):
    if geom["type"] == "Polygon":
        return [geom["coordinates"]]
    return geom["coordinates"]  # MultiPolygon

def draw_choropleth(value_by_zone, titre, fname, cmap, label, fmt="{:.0f}"):
    fig, ax = plt.subplots(figsize=(9.5, 8))
    vals = [v for v in value_by_zone.values()]
    norm = mcolors.Normalize(vmin=min(vals), vmax=max(vals))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    for feat in geo["features"]:
        name = feat["properties"]["shapeName"]
        zone = ZONE_MAP.get(name)
        val = value_by_zone.get(zone)
        color = sm.to_rgba(val) if val is not None else "lightgrey"
        best_area, best_c = -1, None
        for poly in rings_of(feat["geometry"]):
            ext = np.array(poly[0])
            ax.add_patch(MplPoly(ext, closed=True, facecolor=color,
                                 edgecolor="white", linewidth=0.7))
            # centroïde approx du plus grand anneau pour placer l'étiquette
            x, y = ext[:,0], ext[:,1]
            area = abs(np.sum(x*np.roll(y,1) - np.roll(x,1)*y)) / 2
            if area > best_area:
                best_area, best_c = area, (x.mean(), y.mean())
        if best_c is not None:
            txt = name if val is None else f"{name}\n{fmt.format(val)}"
            ax.annotate(txt, best_c, ha="center", va="center", fontsize=7,
                        color="black", weight="bold")
    ax.autoscale(); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(titre, fontsize=13)
    cbar = fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.01); cbar.set_label(label)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, fname), bbox_inches="tight", dpi=120)
    plt.close(fig); print("→", fname)

# Indicateur 1 : inflation moyenne sur les 12 derniers mois par zone
last = ihpc_reg["date"].max()
rec = ihpc_reg[ihpc_reg["date"] > last - pd.DateOffset(months=12)]
infl_zone = rec.groupby("region")["var_annuelle_pct"].mean().to_dict()
draw_choropleth(infl_zone, "Inflation moyenne sur 12 mois par région (%)",
                "14_carte_inflation.png", "YlOrRd", "Inflation (%)", "{:.1f}%")

# Indicateur 2 : coût moyen du panier (12 derniers mois) par zone
recp = panier_reg[panier_reg["date"] > last - pd.DateOffset(months=12)]
panier_zone = (recp.groupby("region")["cout_panier"].mean()/1000).to_dict()
draw_choropleth(panier_zone, "Coût moyen du panier de base par région (milliers FCFA/mois)",
                "15_carte_panier.png", "viridis", "Coût (k FCFA)", "{:.0f}k")


### Synthèse EDA
- L'inflation est restée faible (< 3 %) jusqu'en 2021, a explosé en **2022**
  (pic **+14 %**) sous l'effet conjugué des prix alimentaires mondiaux et de
  l'énergie, puis a reflué (désinflation 2023-2024).
- Les produits **alimentaires de base** (huile, sucre, céréales) et l'**énergie**
  (carburants) sont les plus inflationnistes.
- Les régions **du Sud et du Centre** (Kolda, Diourbel, Kaolack) subissent une
  inflation un peu plus forte que **Dakar**.
- Le **pouvoir d'achat** s'est nettement érodé sur 2022-2023.
